# Day 7 Project Solution: Long-Document Summarizer CLI

Complete Map-Reduce summarization pipeline.
Concepts demonstrated: context-window measurement, overlapping chunks,
`summarize_chunk`, map phase, reduce phase.

## Imports and Constants

In [ ]:
import ollama

MODEL = "llama3.2"

# Lesson 1: conservative context-window estimate for llama3.2.
# 1 token ≈ 4 characters; reserve 200 tokens for the system prompt.
CONTEXT_WINDOW_TOKENS  = 2048
CHARS_PER_TOKEN        = 4
PROMPT_OVERHEAD_TOKENS = 200

MAX_DOC_TOKENS = CONTEXT_WINDOW_TOKENS - PROMPT_OVERHEAD_TOKENS  # 1848 tokens
MAX_DOC_CHARS  = MAX_DOC_TOKENS * CHARS_PER_TOKEN                # ~7 392 chars

# Map-phase prompt: faithfully summarise one passage, nothing added.
SUMMARIZE_SYSTEM_PROMPT = (
    "You are a precise summarization assistant. "
    "The user will send you a passage of text — possibly an excerpt from a "
    "larger document. Summarize it in 2-3 sentences of plain prose. "
    "Do not add headings, bullet points, or information that is not present "
    "in the passage. Use only what is written."
)

# Reduce-phase prompt: synthesise multiple summaries into one coherent paragraph.
REDUCE_SYSTEM_PROMPT = (
    "You are a precise summarization assistant. "
    "You will receive several intermediate summaries, each covering a different "
    "section of a longer document. Synthesise them into one coherent summary "
    "paragraph of 3-5 sentences that covers the document as a whole. "
    "Write plain prose only — no headings, no bullet points, no section labels. "
    "Do not introduce information that is not present in the provided summaries."
)

## Lesson 1 — Measurement: `estimate_tokens` and `will_fit`

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough token count for English prose: 1 token ≈ 4 characters."""
    return len(text) // CHARS_PER_TOKEN


def will_fit(text: str) -> bool:
    """Return True if text is likely within the model's context window."""
    return estimate_tokens(text) <= MAX_DOC_TOKENS


# Quick check
short = "The quarterly results exceeded expectations across all regions."
long  = "A" * 10_000
print(f"Short ({len(short):>5} chars, ~{estimate_tokens(short)} tokens) fits: {will_fit(short)}")
print(f"Long  ({len(long):>5} chars, ~{estimate_tokens(long)} tokens) fits: {will_fit(long)}")

## Lesson 2 — Chunking: `chunk_text`

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """
    Split text into fixed-size chunks with a shared overlap region.

    Args:
        text:       The document to split.
        chunk_size: Maximum number of characters per chunk.
        overlap:    Characters shared between consecutive chunks.
                    Must be strictly less than chunk_size.

    Returns:
        A list of non-empty strings. Returns [] when text is empty.
    """
    if not text:
        return []
    if overlap >= chunk_size:
        raise ValueError(
            f"overlap ({overlap}) must be less than chunk_size ({chunk_size})"
        )

    # step < chunk_size ensures the cursor always advances
    step = chunk_size - overlap
    chunks: list[str] = []
    cursor = 0

    while cursor < len(text):
        chunks.append(text[cursor : cursor + chunk_size])
        cursor += step

    return chunks


# Smoke test on a short alphabet string
pieces = chunk_text("ABCDEFGHIJKLMNOPQRSTUVWXYZ", chunk_size=10, overlap=3)
for i, p in enumerate(pieces):
    print(f"  Chunk {i}: '{p}'")

## Lesson 3 — Map Atom: `summarize_chunk`

In [ ]:
def summarize_chunk(chunk: str, model: str = MODEL) -> str:
    """
    Summarize a single text chunk using a local Ollama model.

    Args:
        chunk: A string that already fits within the model's context window.
        model: Ollama model name.

    Returns:
        A plain-prose summary string (2-3 sentences).
    """
    # System prompt ensures consistent, constrained output on every call
    messages = [
        {"role": "system", "content": SUMMARIZE_SYSTEM_PROMPT},
        {"role": "user",   "content": chunk},
    ]
    response = ollama.chat(model=model, messages=messages)
    return response["message"]["content"]

## Lesson 4 — Map Phase: `map_summaries`

In [ ]:
def map_summaries(chunks: list[str], model: str = MODEL) -> list[str]:
    """
    Apply summarize_chunk() to every chunk and collect the results.

    Args:
        chunks: List of text strings from chunk_text(). Each must fit the context window.
        model:  Ollama model name.

    Returns:
        A list of summary strings, one per chunk, in the same order.
        len(result) == len(chunks) is always true.
    """
    summaries: list[str] = []
    total = len(chunks)
    for i, chunk in enumerate(chunks, start=1):
        print(f"  Summarizing chunk {i}/{total} ({len(chunk)} chars)...")
        summaries.append(summarize_chunk(chunk, model=model))
    return summaries

## Lesson 5 — Reduce Phase: `reduce_summaries`

In [ ]:
def reduce_summaries(chunk_summaries: list[str], model: str = MODEL) -> str:
    """
    Combine a list of per-chunk summaries into one final summary.

    Sends all summaries in a single ollama.chat() call — always exactly one
    reduce call regardless of how many chunks the map phase produced.

    Args:
        chunk_summaries: List of short summary strings from the map phase.
        model:           Ollama model name.

    Returns:
        A single coherent summary paragraph covering the whole document.
    """
    # Double newline lets the model see each intermediate summary as a distinct block
    combined = "\n\n".join(chunk_summaries)
    messages = [
        {"role": "system", "content": REDUCE_SYSTEM_PROMPT},
        {"role": "user",   "content": combined},
    ]
    response = ollama.chat(model=model, messages=messages)
    return response["message"]["content"]

## Lessons 4 + 5 — Full Pipeline: `summarize_document`

One string in, one string out. The caller never sees chunks, intermediate summaries,
or model calls — only a clean final result.

In [ ]:
def summarize_document(
    text: str,
    chunk_size: int = 500,
    overlap: int = 50,
    model: str = MODEL,
) -> str:
    """
    Summarize a document of any length using Map-Reduce.

    Short documents (fit in one context window) go directly to the model.
    Long documents are: chunked -> each chunk summarised (Map) ->
    all chunk summaries synthesised into one final result (Reduce).

    Prints each chunk summary as it is produced (map phase progress),
    then prints the final summary (reduce phase result).

    Args:
        text:       The full document text to summarise.
        chunk_size: Maximum characters per chunk (default 500).
        overlap:    Characters shared between consecutive chunks (default 50).
        model:      Ollama model name (default 'llama3.2').

    Returns:
        A single summary string covering the entire document.
    """
    # Fast path: document fits in one model call — no chunking needed
    if will_fit(text):
        print("Document fits context window — summarizing in one call.")
        result = summarize_chunk(text, model=model)
        print("\n[Direct summary]")
        print(result)
        return result

    # Split the document into overlapping chunks (Lesson 2)
    chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
    total  = len(chunks)
    print(f"Document too long for one call ({len(text)} chars).")
    print(f"Split into {total} chunks (chunk_size={chunk_size}, overlap={overlap}).")
    print()

    # Map phase: summarize each chunk via map_summaries (Lessons 3 & 4)
    print("--- MAP PHASE ---")
    chunk_summaries = map_summaries(chunks, model=model)
    for i, summary in enumerate(chunk_summaries, start=1):
        print(f"[Chunk {i} summary]")
        print(summary)
        print()

    # Reduce phase: one final call collapses all summaries (Lesson 5)
    print("--- REDUCE PHASE ---")
    print(f"Synthesising {total} chunk summaries into one final summary...")
    final = reduce_summaries(chunk_summaries, model=model)
    return final

## Run the Pipeline

The sample document is ~8 400 characters — above the model's practical context limit
(~7 392 chars), so the pipeline splits it into 2 chunks (chunk_size=5 000, overlap=100)
and runs one map call per chunk followed by one reduce call.

In [ ]:
# Sample long document — repeated paragraph forces the map-reduce path
SAMPLE_TEXT = (
    "Global temperatures have risen steadily over the past century due to "
    "greenhouse gas emissions from industrial activity and deforestation. "
    "Scientists have documented melting ice caps, rising sea levels, and "
    "increasingly severe weather events as consequences of this warming trend. "
) * 30  # ~8 400 chars — above the 7 392-char context limit, forces map-reduce

print(f"Document length: {len(SAMPLE_TEXT)} characters")
print(f"Fits in one call: {will_fit(SAMPLE_TEXT)}")
print()

final_summary = summarize_document(
    SAMPLE_TEXT,
    chunk_size=5000,   # 2 chunks → 2 map calls + 1 reduce call
    overlap=100,
    model="llama3.2",
)

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(final_summary)

## Deliverable Confirmation

In [ ]:
# Verify the pipeline produced a non-empty final summary
assert isinstance(final_summary, str), "final_summary must be a string"
assert len(final_summary) > 0,        "final_summary must not be empty"

# Verify intermediate summaries were produced (map phase ran)
# Re-run a lightweight version that collects them for assertion
chunks = chunk_text(SAMPLE_TEXT, chunk_size=500, overlap=50)
assert len(chunks) > 1, "sample text should have produced more than one chunk"

print(f"Document length  : {len(SAMPLE_TEXT)} chars")
print(f"Chunks produced  : {len(chunks)}")
print(f"Final summary    : {len(final_summary)} chars")
print()
print("Deliverable confirmed: summarize_document() ran the full Map-Reduce pipeline.")
print("- Map phase   : printed each chunk summary as it was produced.")
print("- Reduce phase: produced one final coherent summary from all chunk summaries.")
print()
print("Day 7 project complete.")